# Classification examples

In [ ]:
from sklearn.datasets import load_wine
import itertools
import sys

sys.path.append("..")

from ai_toolkit import (
    ConfigFactory,
    BaseDataset,
    LogisticRegressionModel,
    NaiveBayesModel,
    XGBoostModel,
    get_all_classification_models, 
    ClassificationModelTrainer, 
    lazypredict_classification,
    EnsembleVotingClassifierModel,
    EnsembleStackingClassifierModel,
    display_banner,
)

In [ ]:
display_banner()

## Example data

In [ ]:
class ClfDataset(BaseDataset):
    """Dataset for wine multi-class classification task.
    https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html
    """

    def __init__(self):
        """Initialize the clf dataset."""

        super().__init__()

    def load_data(self):
        """Load the wine dataset for classification task."""
        
        self.X, self.y = load_wine(return_X_y=True, as_frame=True)
        self.X_test = self.X.head()

In [ ]:
CDataset = ClfDataset()
CDataset.load_data()
CDataset.preprocess()
X_clf, y_clf, X_test_clf = CDataset.get_data()

## Configuration

In [ ]:
config_factory = ConfigFactory()
configs = config_factory.get_config()

# config_factory.save_config_file("config_development.yaml")
config = configs.training

## Training and evaluation

### Train one example model

In [ ]:
base_model = LogisticRegressionModel()

# Create a classification model trainer
trainer = ClassificationModelTrainer(
    base_model=base_model,
    config_factory=config_factory,
)

In [ ]:
# Train and optimize the model
best_model, mean_metrics = trainer.train_and_optimize(
    X=X_clf, 
    y=y_clf, 
)

# Predict on the test set
# y_pred, y_pred_proba = trainer.predict(X_test_clf)

### Train all classification models

In [ ]:
base_models = get_all_classification_models()

for base_model in base_models.values():

    # Create a classification model trainer
    trainer = ClassificationModelTrainer(
        base_model=base_model,
        config_factory=config_factory,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_clf, 
        y=y_clf, 
    )

    # Predict on the test set
    # y_pred, y_pred_proba = trainer.predict(X_test_clf)

In [ ]:
df_mean_results = lazypredict_classification(
    X=X_clf, 
    y=y_clf, 
    n_splits=config.n_splits,
    random_state=config.random_state,
)

# print(df_mean_results.to_string())
df_mean_results

### Ensemble

In [ ]:
# (Model, mlflow run_id) pairs
model_pool = [
    (LogisticRegressionModel(), "0f1ed6332d7546a8b40fb8bdd1f3176c"),
    (NaiveBayesModel(), "54e32a7a289d46488ae01625119ff369"),
    (XGBoostModel(), "40e33107c2274c969e99e5d09bf786aa"),
]

meta_model = LogisticRegressionModel()

In [ ]:
combinations = []
 
for model in range(2, len(model_pool) + 1):
    combinations.extend(itertools.combinations(model_pool, model))

#### Voting | Train all combinations

In [ ]:
for combination in combinations:
    models = list(combination)

    base_model = EnsembleVotingClassifierModel(
        models=models,
    )

    # Create a classification model trainer
    trainer = ClassificationModelTrainer(
        base_model=base_model,
        config_factory=config_factory,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_clf, 
        y=y_clf, 
    )

    # Predict on the test set
    # y_pred, y_pred_proba = trainer.predict(X_test_clf)

#### Stacking | Train all combinations

In [ ]:
for combination in combinations:
    models = list(combination)
    
    base_model = EnsembleStackingClassifierModel(
        models=models,
        meta_model=meta_model,
    )

    # Create a classification model trainer
    trainer = ClassificationModelTrainer(
        base_model=base_model,
        config_factory=config_factory,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_clf, 
        y=y_clf, 
    )

    # Predict on the test set
    # y_pred, y_pred_proba = trainer.predict(X_test_clf)